# Mesh Simplification Algorithms 

In [1]:
import vtk

In [2]:
object_path = "./assets/cat1.obj"

# --- Load OBJ file ---
reader = vtk.vtkOBJReader()
reader.SetFileName(object_path)
reader.Update()
mesh = reader.GetOutput()


In [3]:
def show_mesh(mesh, title="Mesh"):
    mapper = vtk.vtkPolyDataMapper()
    mapper.SetInputData(mesh)
    
    actor = vtk.vtkActor()
    actor.SetMapper(mapper)
    
    renderer = vtk.vtkRenderer()
    renderer.AddActor(actor)
    renderer.SetBackground(0.1, 0.1, 0.1)
    
    window = vtk.vtkRenderWindow()
    window.AddRenderer(renderer)
    window.SetWindowName(title)
    window.SetSize(800, 800)
    
    interactor = vtk.vtkRenderWindowInteractor()
    interactor.SetRenderWindow(window)
    
    window.Render()
    interactor.Start()

In [4]:
# --- Helper: print mesh statistics ---
def mesh_stats(mesh, name="Mesh"):
    num_points = mesh.GetNumberOfPoints()
    num_polys = mesh.GetNumberOfPolys()
    num_lines = mesh.GetNumberOfLines()  # for wireframe-like meshes
    print(f"📦 {name}")
    print(f"   Vertices (Points): {num_points}")
    print(f"   Faces (Polys):     {num_polys}")
    print()

In [5]:
show_mesh(mesh, "Original Mesh" )

mesh_stats(mesh, "Original Mesh")

📦 Original Mesh
   Vertices (Points): 4008
   Faces (Polys):     8012



## Mesh Simplification Algorithms Overview

This notebook applies several mesh simplification techniques each with different strategies to reduce the number of vertices and faces while preserving the overall shape of a 3D object.

---

### Vertex Clustering (`vtkQuadricClustering`)

**Idea:**  
Divide the 3D space into a uniform grid. All vertices that fall into the same cell are *merged* into a single representative vertex (usually the cell’s centroid).

**How it simplifies:**  
- Reduces vertex count by clustering close points.  
- Fast and suitable for large meshes.  
- May introduce noticeable artifacts on fine details due to uniform grid merging.

In [6]:
# --- Vertex Clustering ---
cluster = vtk.vtkQuadricClustering()
cluster.SetInputData(mesh)
cluster.SetNumberOfXDivisions(40)
cluster.SetNumberOfYDivisions(40)
cluster.SetNumberOfZDivisions(40)
cluster.Update()
clustered = cluster.GetOutput()

show_mesh(clustered, "Vertex Clustering")
mesh_stats(clustered, "Clustered Mesh")

📦 Clustered Mesh
   Vertices (Points): 955
   Faces (Polys):     2004



### Vertex Removal (`vtkCleanPolyData`)

**Idea:**
Remove duplicate, isolated, or unused vertices that don’t contribute to any polygon or that are within a small spatial tolerance.

**How it simplifies:**
- Cleans redundant geometry.
- Slightly reduces mesh complexity.
- Used as a pre-processing or cleanup step before deeper simplification.

In [32]:
# --- vertex Removal ---
def vertex_removal(mesh, tolerance=0.001):
    """Simulate vertex removal by cleaning redundant vertices."""
    clean = vtk.vtkCleanPolyData()
    clean.SetInputData(mesh)
    clean.SetTolerance(tolerance)
    clean.PointMergingOn()
    clean.Update()
    return clean.GetOutput()


cleaned = vertex_removal(clustered)
show_mesh(cleaned, "Vertex Removal")
mesh_stats(cleaned, "Cleaned Mesh")

📦 Cleaned Mesh
   Vertices (Points): 955
   Faces (Polys):     2004



## Edge Collapse (`vtkDecimatePro`)

**Idea:**
Iteratively remove edges and merge their connected vertices when the resulting geometric error is below a threshold.

How it simplifies:
- Collapses less important edges while keeping the general surface shape.
- Reduces both vertices and faces effectively.
- Balances accuracy and performance.

In [33]:
# --- Edge Collapse (Decimation) ---
decimate = vtk.vtkDecimatePro()
decimate.SetInputData(mesh)
decimate.SetTargetReduction(0.8)  # 80% reduction
decimate.PreserveTopologyOn()
decimate.Update()
collapsed = decimate.GetOutput()

show_mesh(collapsed, "Edge Collapse (DecimatePro)")
mesh_stats(collapsed, "Collapsed Mesh")

📦 Collapsed Mesh
   Vertices (Points): 1564
   Faces (Polys):     3124



## Half-Edge Collapse (`vtkQuadricDecimation`)

**Idea:**
Uses quadric error metrics to evaluate the geometric cost of collapsing edges.
Each vertex stores a quadric matrix representing its distance from surrounding planes; the algorithm merges vertices to minimize total error.

How it simplifies:

- More accurate and smooth results compared to simple edge collapse.
- Preserves sharp features better.
- Ideal for high-quality mesh decimation.

In [34]:
# ---  Half-Edge Collapse (similar effect using vtkQuadricDecimation) ---
quadric_dec = vtk.vtkQuadricDecimation()
quadric_dec.SetInputData(mesh)
quadric_dec.SetTargetReduction(0.8)
quadric_dec.Update()
half_edge = quadric_dec.GetOutput()

show_mesh(half_edge, "Half-Edge Collapse (Quadric Decimation)")
mesh_stats(half_edge, "Half-Edge Mesh")

📦 Half-Edge Mesh
   Vertices (Points): 803
   Faces (Polys):     1601



### Simplification algorithms Combined

In [37]:
# --- Combine Algorithms: Vertex Clustering → Edge Collapse ---
combo1 = vtk.vtkDecimatePro()
combo1.SetInputData(clustered)
combo1.SetTargetReduction(0.3)
combo1.Update()
combo1_result = combo1.GetOutput()

show_mesh(combo1_result, "Cluster → Edge Collapse")
mesh_stats(combo1_result,"Combination Vertex Clustering → Edge Collapse")

📦 Combination Vertex Clustering → Edge Collapse
   Vertices (Points): 653
   Faces (Polys):     1402



In [38]:
# --- Combine Algorithms: Edge Collapse → Vertex Clustering ---
combo2 = vtk.vtkQuadricClustering()
combo2.SetInputData(collapsed)
combo2.SetNumberOfXDivisions(40)
combo2.SetNumberOfYDivisions(40)
combo2.SetNumberOfZDivisions(40)
combo2.Update()
combo2_result = combo2.GetOutput()

show_mesh(combo2_result, "Edge Collapse → Clustering")
mesh_stats(combo2_result,"Edge Collapse → Clustering")

📦 Edge Collapse → Clustering
   Vertices (Points): 400
   Faces (Polys):     833



In [39]:
# --- First Vertex Clustering Pass ---
cluster1 = vtk.vtkQuadricClustering()
cluster1.SetInputData(mesh)
cluster1.SetNumberOfXDivisions(30)
cluster1.SetNumberOfYDivisions(30)
cluster1.SetNumberOfZDivisions(30)
cluster1.Update()
mesh_clustered_once = cluster1.GetOutput()

# show_mesh(mesh_clustered_once, "Vertex Clustering - 1st Pass")

# --- Second Vertex Clustering Pass ---
cluster2 = vtk.vtkQuadricClustering()
cluster2.SetInputData(mesh_clustered_once)
cluster2.SetNumberOfXDivisions(15)  # fewer divisions = coarser grid
cluster2.SetNumberOfYDivisions(15)
cluster2.SetNumberOfZDivisions(15)
cluster2.Update()
mesh_clustered_twice = cluster2.GetOutput()

show_mesh(mesh_clustered_twice, "Vertex Clustering - 2nd Pass")
mesh_stats(mesh_clustered_twice, "Vertex Clustering - 2nd Pass")


📦 Vertex Clustering - 2nd Pass
   Vertices (Points): 343
   Faces (Polys):     717



In [40]:
# --- Vertex Clustering + Vertex Removal ---

# --- Vertex Clustering ---
cluster = vtk.vtkQuadricClustering()
cluster.SetInputData(mesh)
cluster.SetNumberOfXDivisions(30)
cluster.SetNumberOfYDivisions(30)
cluster.SetNumberOfZDivisions(30)
cluster.Update()
clustered = cluster.GetOutput()

# --- Vertex Removal ---
cleaned = vertex_removal(clustered)


show_mesh(cleaned, "Vertex Clustering + Vertex Removal")
mesh_stats(cleaned, "Vertex Clustering + Vertex Removal")

📦 Vertex Clustering + Vertex Removal
   Vertices (Points): 955
   Faces (Polys):     2004

